# Required imports
Always run

In [1]:
import os
import glob

import numpy as np
import matplotlib.pyplot as plt

import librosa
from scipy.fftpack import dct

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv1D, MaxPooling1D, Flatten, GaussianNoise, Reshape
from tensorflow.keras.layers import GlobalAveragePooling1D

# Load dataset and extract MFCC features

In [ ]:
"""
feature_extraction.py
=====================
Replica Python della pipeline DSP di micro_speech_dsp.ino
(TF Lite Micro Speech – Arduino Nano 33 BLE Sense).

Pipeline per ogni clip WAV:
  PCM 16 kHz → frame 25 ms / stride 20 ms → FFT 512 pt
  → Mel filterbank 32 ch (125–7500 Hz)
  → Noise reduction SRNN
  → PCAN gain control
  → Log scale
  → Quantizzazione int8  →  output (49, 32) int8

Output finale pronto per il training TensorFlow:
  X_train / X_test : np.ndarray  (N, 49, 32)  dtype=int8
  y_train / y_test : tf.Tensor   (N, num_classes)  one-hot int64
"""

# ─────────────────────────────────────────────────────────────────────────────
# Dipendenze
# ─────────────────────────────────────────────────────────────────────────────
import os
import glob
import numpy as np
import librosa
import tensorflow as tf
from tqdm import tqdm
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv1D, MaxPooling1D, Flatten, GaussianNoise, Reshape
from tensorflow.keras.layers import GlobalAveragePooling1D


# ─────────────────────────────────────────────────────────────────────────────
# Costanti del DSP  (identiche al .ino)
# ─────────────────────────────────────────────────────────────────────────────

SAMPLE_RATE         = 16_000     # Hz
FRAME_LEN_MS        = 25         # ms per slice  → 400 campioni
FRAME_STEP_MS       = 20         # ms di stride  → 320 campioni
N_MEL               = 32         # canali filterbank
FMIN                = 125.0      # Hz
FMAX                = 7_500.0    # Hz
N_SLICES            = 49         # slice nella finestra da 1 s
FFT_SIZE            = 512        # prossima pot. di 2 ≥ 400

FRAME_LEN           = FRAME_LEN_MS  * SAMPLE_RATE // 1000  # 400
FRAME_STEP          = FRAME_STEP_MS * SAMPLE_RATE // 1000  # 320

# Verifica: (16000 − 400) // 320 + 1 = 49  ✓
assert (SAMPLE_RATE - FRAME_LEN) // FRAME_STEP + 1 == N_SLICES, \
    "Mismatch numero di slices: controlla SAMPLE_RATE / FRAME_LEN / FRAME_STEP"

# Noise reduction (noise_reduction.c)
EVEN_SMOOTH         = 0.025
ODD_SMOOTH          = 0.06
MIN_SIG_REMAIN      = 0.05

# PCAN gain control (pcan_gain_control.c)
PCAN_STRENGTH       = 0.95
PCAN_OFFSET         = 80.0
PCAN_GAIN_BITS      = 21

# Log scale (log_scale.c)
LOG_SCALE_SHIFT     = 6          # shift prima del log  → divide per 64
LOG_SCALE_OUT_BITS  = 12         # bit dell'output      → moltiplica per 4096

# Quantizzazione int8  (GenerateMicroFeatures nel .ino)
#   raw uint16 ∈ [0, ~670]  →  int8 ∈ [-128, 127]
#   formula: value = (raw × 256 + 333) / 666 − 128
K_VALUE_SCALE       = 256
K_VALUE_DIV         = int(25.6 * 26.0 + 0.5)   # 666


# ─────────────────────────────────────────────────────────────────────────────
# Mel filterbank  (calcolato una sola volta all'avvio)
# ─────────────────────────────────────────────────────────────────────────────
# tf.signal.linear_to_mel_weight_matrix usa la stessa scala HTK del
# microfrontend TF Lite; restituisce shape (257, 32) già trasposta per @.
_MEL_FILTERBANK: np.ndarray = tf.signal.linear_to_mel_weight_matrix(
    num_mel_bins          = N_MEL,
    num_spectrogram_bins  = FFT_SIZE // 2 + 1,   # 257
    sample_rate           = SAMPLE_RATE,
    lower_edge_hertz      = FMIN,
    upper_edge_hertz      = FMAX,
    dtype                 = tf.float32,
).numpy()   # (257, 32)

# Finestra di Hann per i frame da 400 campioni (identica all'implementazione C)
_HANN_WINDOW: np.ndarray = np.hanning(FRAME_LEN).astype(np.float32)


# ─────────────────────────────────────────────────────────────────────────────
# Pipeline DSP
# ─────────────────────────────────────────────────────────────────────────────

def _power_spectrum(frames: np.ndarray) -> np.ndarray:
    """
    Applica finestra di Hann, zero-padding a 512 pt, FFT e spettro di potenza.

    Parametri
    ---------
    frames : (N_SLICES, FRAME_LEN) float32
        Campioni PCM scalati in range int16 (×32768).

    Ritorna
    -------
    (N_SLICES, 257) float32 – modulo quadro dei bin FFT mono-laterali.
    """
    windowed = frames * _HANN_WINDOW                          # (49, 400)
    padded   = np.zeros((N_SLICES, FFT_SIZE), dtype=np.float32)
    padded[:, :FRAME_LEN] = windowed
    spectra  = np.abs(np.fft.rfft(padded, axis=1)) ** 2      # (49, 257)
    return spectra


def _mel_energy(spectra: np.ndarray) -> np.ndarray:
    """
    Proietta lo spettro di potenza nei canali mel.

    Parametri
    ---------
    spectra : (N_SLICES, 257) float32

    Ritorna
    -------
    (N_SLICES, N_MEL) float32 – energia per canale mel (valori assoluti grandi).
    """
    return spectra @ _MEL_FILTERBANK   # (49, 32)


def _noise_reduction_and_pcan(mel: np.ndarray) -> np.ndarray:
    """
    Noise reduction SRNN + PCAN gain control frame per frame.

    Equivale all'esecuzione in sequenza di:
      NoiseReductionApply()  →  aggiorna noise_estimate, produce segnale ripulito
      PcanGainControlApply() →  normalizza usando lo stesso noise_estimate

    Parametri
    ---------
    mel : (N_SLICES, N_MEL) float32

    Ritorna
    -------
    (N_SLICES, N_MEL) float32 – output PCAN con valori tipicamente ∈ [0, ~20].
    """
    noise_est  = np.zeros(N_MEL, dtype=np.float64)
    pcan_out   = np.zeros_like(mel)

    for i in range(N_SLICES):
        # ── Noise reduction ──────────────────────────────────────
        # Il microfrontend C usa smoothing diverso per frame pari/dispari.
        alpha = EVEN_SMOOTH if (i % 2 == 0) else ODD_SMOOTH
        noise_est = (1.0 - alpha) * noise_est + alpha * mel[i].astype(np.float64)

        sig      = mel[i].astype(np.float64)
        denoised = np.maximum(
            sig - (1.0 - MIN_SIG_REMAIN) * noise_est,
            MIN_SIG_REMAIN * sig
        )

        # ── PCAN gain control ────────────────────────────────────
        # output = denoised / (noise_est^strength + offset)
        # Lo stesso noise_estimate usato sopra, come in pcan_gain_control.c
        denom         = np.power(np.maximum(noise_est, 1e-12), PCAN_STRENGTH) + PCAN_OFFSET
        pcan_out[i]   = (denoised / denom).astype(np.float32)

    return pcan_out


def _log_scale(pcan: np.ndarray) -> np.ndarray:
    """
    Log-scale compression (log_scale.c, scale_shift=6, out_bits=12).

      output = log2(1 + x / 2^scale_shift) × 2^out_bits
             = log2(1 + x / 64) × 4096

    Produce valori uint16 tipicamente ∈ [0, ~670].

    Parametri
    ---------
    pcan : (N_SLICES, N_MEL) float32

    Ritorna
    -------
    (N_SLICES, N_MEL) float32
    """
    scale_in  = float(1 << LOG_SCALE_SHIFT)    # 64.0
    scale_out = float(1 << LOG_SCALE_OUT_BITS)  # 4096.0
    return np.log2(1.0 + np.maximum(pcan, 0.0) / scale_in) * scale_out


def _quantize_int8(log_out: np.ndarray) -> np.ndarray:
    """
    Quantizzazione uint16 → int8 identica a GenerateMicroFeatures() nel .ino.

      value = (raw × 256 + 333) / 666 − 128
      clamp a [-128, 127]

    Parametri
    ---------
    log_out : (N_SLICES, N_MEL) float32  – valori raw ∈ [0, ~670]

    Ritorna
    -------
    (N_SLICES, N_MEL) int8
    """
    raw   = log_out.astype(np.int64)
    value = (raw * K_VALUE_SCALE + K_VALUE_DIV // 2) // K_VALUE_DIV - 128
    return np.clip(value, -128, 127).astype(np.int8)


def extract_features(wav_path: str) -> np.ndarray:
    """
    Carica un file WAV e restituisce il tensore di feature int8.

    L'output è identico al buffer g_feature_data[kFeatureElementCount]
    dell'Arduino (49 × 32 = 1568 byte).

    Parametri
    ---------
    wav_path : str – percorso al file .wav (qualsiasi sample rate / canali).

    Ritorna
    -------
    (49, 32) int8 – spectrogram quantizzato.
    """
    # ── Carica e normalizza l'audio ───────────────────────────────────────────
    # librosa resampla a 16 kHz e converte in mono automaticamente.
    audio, _ = librosa.load(wav_path, sr=SAMPLE_RATE, mono=True, duration=1.0)

    # Padding / trimming per garantire esattamente 1 s = 16000 campioni
    target_len = SAMPLE_RATE
    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))
    else:
        audio = audio[:target_len]

    # Scala a range int16 come i campioni PDM dell'Arduino
    audio = (audio * 32768.0).astype(np.float32)

    # ── Framing ───────────────────────────────────────────────────────────────
    # sliding_window_view + stride manuale = equivalente di librosa.util.frame
    frames = np.lib.stride_tricks.sliding_window_view(audio, FRAME_LEN)[::FRAME_STEP]
    frames = frames[:N_SLICES].copy()   # (49, 400)

    # ── DSP pipeline ─────────────────────────────────────────────────────────
    spectra  = _power_spectrum(frames)           # (49, 257) float32
    mel      = _mel_energy(spectra)              # (49, 32)  float32
    pcan     = _noise_reduction_and_pcan(mel)    # (49, 32)  float32
    log_out  = _log_scale(pcan)                  # (49, 32)  float32
    features = _quantize_int8(log_out)           # (49, 32)  int8

    return features


# ─────────────────────────────────────────────────────────────────────────────
# Caricamento dataset (struttura identica al codice nel tuo notebook)
# ─────────────────────────────────────────────────────────────────────────────

def load_dataset(folder: str):
    """
    Scansiona *folder* per file .wav.
    Il label è la parte del filename prima del primo punto.

    Ritorna
    -------
    file_paths : list[str]
    labels     : list[str]
    """
    file_paths, labels = [], []
    for file in glob.glob(os.path.join(folder, "*.wav")):
        label = os.path.basename(file).split(".")[0]
        file_paths.append(file)
        labels.append(label)
    return file_paths, labels


def process_split(file_paths: list, split_name: str) -> np.ndarray:
    """
    Estrae le feature da tutti i file di un split con barra di avanzamento.

    Parametri
    ---------
    file_paths : list[str]
    split_name : str – usato solo per il titolo della progress bar

    Ritorna
    -------
    (N, 49, 32) int8
    """
    features = []
    errors   = []

    for path in tqdm(file_paths, desc=f"Extracting {split_name}", unit="file"):
        try:
            features.append(extract_features(path))
        except Exception as exc:
            errors.append((path, str(exc)))
            # Inserisce un tensore di zeri per non rompere l'allineamento con y
            features.append(np.zeros((N_SLICES, N_MEL), dtype=np.int8))

    if errors:
        print(f"\n[WARN] {len(errors)} file non processati in {split_name}:")
        for p, e in errors[:5]:
            print(f"  {p}: {e}")
        if len(errors) > 5:
            print(f"  … e altri {len(errors) - 5}")

    return np.stack(features, axis=0)   # (N, 49, 32)


# ─────────────────────────────────────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    dataset_path = "dataset"
    train_dir    = os.path.join(dataset_path, "training")
    test_dir     = os.path.join(dataset_path, "testing")

    # ── Carica percorsi e label testuali ─────────────────────────────────────
    X_train_paths, y_train_str = load_dataset(train_dir)
    X_test_paths,  y_test_str  = load_dataset(test_dir)

    print(f"Training samples : {len(X_train_paths)}")
    print(f"Testing samples  : {len(X_test_paths)}")

    # ── StringLookup → one-hot (identico al notebook originale) ──────────────
    lookup = tf.keras.layers.StringLookup(
        output_mode    = "one_hot",
        num_oov_indices = 0,
    )
    lookup.adapt(y_train_str)

    y_train = lookup(y_train_str)   # (N_train, num_classes) int64
    y_test  = lookup(y_test_str)    # (N_test,  num_classes) int64

    labels      = lookup.get_vocabulary()
    num_classes = len(labels)
    print(f"Classi ({num_classes}): {labels}")

    # ── Estrazione feature ────────────────────────────────────────────────────
    X_train = process_split(X_train_paths, "train")   # (N_train, 49, 32) int8
    X_test  = process_split(X_test_paths,  "test")    # (N_test,  49, 32) int8

    print(f"\nX_train shape : {X_train.shape}  dtype={X_train.dtype}")
    print(f"X_test  shape : {X_test.shape}   dtype={X_test.dtype}")
    print(f"y_train shape : {y_train.shape}  dtype={y_train.dtype}")
    print(f"y_test  shape : {y_test.shape}   dtype={y_test.dtype}")

    # ── Salva su disco ────────────────────────────────────────────────────────
    os.makedirs("features", exist_ok=True)
    np.save("features/X_train.npy", X_train)
    np.save("features/X_test.npy",  X_test)
    np.save("features/y_train.npy", y_train.numpy())
    np.save("features/y_test.npy",  y_test.numpy())
    np.save("features/labels.npy",  np.array(labels))

    print("\nFeature salvate in features/")
    print("  X_train.npy  X_test.npy  y_train.npy  y_test.npy  labels.npy")

    # ─────────────────────────────────────────────────────────────────────────
    # Costruzione tf.data.Dataset pronti per il training
    # ─────────────────────────────────────────────────────────────────────────

    BATCH_SIZE  = 32
    AUTOTUNE    = tf.data.AUTOTUNE

    # Converti X in float32 (range approssimativo [-1, 1] dopo rescaling)
    # Il modello riceve (49, 32, 1) – aggiunta dim canale per Conv2D.
    def make_dataset(X: np.ndarray, y: tf.Tensor, shuffle: bool) -> tf.data.Dataset:
        # float32 normalizzato in [-1, 1] per facilitare il training
        X_f = (X.astype(np.float32) / 128.0).reshape(len(X), -1)  # (N, 1568)
        ds  = tf.data.Dataset.from_tensor_slices((X_f, y))
        if shuffle:
            ds = ds.shuffle(buffer_size=len(X), reshuffle_each_iteration=True)
        return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

    train_ds = make_dataset(X_train, y_train, shuffle=True)
    test_ds  = make_dataset(X_test,  y_test,  shuffle=False)

    print("\ntrain_ds:", train_ds)
    print("test_ds :", test_ds)

    # ─────────────────────────────────────────────────────────────────────────
    # Esempio di modello CNN compatibile con il tensore (49, 32, 1)
    # (opzionale – decommentare per un training di verifica)
    # ─────────────────────────────────────────────────────────────────────────

    model = Sequential()
    model.add(tf.keras.Input(shape=(1568,)))
    model.add(GaussianNoise(0.1))
    model.add(Reshape((49, 32)))
    model.add(Conv1D(8, kernel_size=5, padding='same', activation='relu'))
    model.add(MaxPooling1D(pool_size=2, strides=2, padding='same'))
    model.add(Dropout(0.25))
    model.add(Conv1D(16, kernel_size=5, padding='same', activation='relu'))
    model.add(MaxPooling1D(pool_size=2, strides=2, padding='same'))
    model.add(Dropout(0.25))
    model.add(GlobalAveragePooling1D())
    model.add(Dense(num_classes, activation='softmax')) 
    
    model.compile(
        optimizer = "adam",
        loss      = "categorical_crossentropy",
        metrics   = ["accuracy"],
    )
    model.summary()
    
    history = model.fit(
        train_ds,
        validation_data = test_ds,
        epochs          = 100,
    )

Training samples : 2615
Testing samples  : 641
Classi (5): [np.str_('noise'), np.str_('unknown'), np.str_('heynano'), np.str_('on'), np.str_('off')]


Extracting test: 100%|██████████| 641/641 [00:01<00:00, 409.87file/s]



X_train shape : (2615, 49, 32)  dtype=int8
X_test  shape : (641, 49, 32)   dtype=int8
y_train shape : (2615, 5)  dtype=<dtype: 'int64'>
y_test  shape : (641, 5)   dtype=<dtype: 'int64'>

Feature salvate in features/
  X_train.npy  X_test.npy  y_train.npy  y_test.npy  labels.npy

train_ds: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 1568), dtype=tf.float32, name=None), TensorSpec(shape=(None, 5), dtype=tf.int64, name=None))>
test_ds : <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 1568), dtype=tf.float32, name=None), TensorSpec(shape=(None, 5), dtype=tf.int64, name=None))>


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gaussian_noise_1                │ (None, 1568)           │             0 │
│ (GaussianNoise)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_2 (Reshape)             │ (None, 49, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, 49, 8)          │         1,288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_4 (MaxPooling1D)  │ (None, 25, 8)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 25, 8)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 25, 16)         │           656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_5 (MaxPooling1D)  │ (None, 13, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 13, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_2      │ (None, 16)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │            85 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,029 (7.93 KB)

 Trainable params: 2,029 (7.93 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.2707 - loss: 1.5792 - val_accuracy: 0.4306 - val_loss: 1.4883
Epoch 2/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4792 - loss: 1.3609 - val_accuracy: 0.6022 - val_loss: 1.1581
Epoch 3/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5962 - loss: 1.0753 - val_accuracy: 0.7098 - val_loss: 0.8909
Epoch 4/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6463 - loss: 0.9102 - val_accuracy: 0.7254 - val_loss: 0.7955
Epoch 5/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.6807 - loss: 0.8254 - val_accuracy: 0.7395 - val_loss: 0.7249
Epoch 6/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.6899 - loss: 0.7935 - val_accuracy: 0.7488 - val_loss: 0.6893
Epoch 7/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.6948 - loss: 0.7709 - val_accuracy: 0.7363 - val_loss: 0.6680
Epoch 8/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7017 - loss: 0.7596 - val_accuracy: 0.758

In [10]:
# Save features
os.makedirs("features1", exist_ok=True)
np.savez("features1/features.npz", X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test)

In [11]:
# Valutazione
test_loss, test_acc = model.evaluate(test_ds)

print("Test accuracy:", test_acc)

21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8268 - loss: 0.4743
Test accuracy: 0.8268330693244934


In [12]:
# Load feature
data = np.load("features1/features.npz")

X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(2615, 49, 32)
(641, 49, 32)
(2615, 5)
(641, 5)


# Quantization

In [13]:
# Load model
# model = tf.keras.models.load_model("models/best.keras")

In [14]:
# Convert to TFLite full int8
def representative_dataset():
    idx = np.random.choice(len(X_train), 300, replace=False)
    for i in idx:
        data = X_train[i].astype(np.float32)
        data = np.expand_dims(data, axis=0)
        yield [data]

converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

converter.representative_dataset = representative_dataset

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_quant_model = converter.convert()
os.makedirs("models1", exist_ok=True)
save_path = "models1/best.tflite"

with open(save_path, "wb") as f:
    f.write(tflite_quant_model)

print("Modello salvato in:", save_path)

INFO:tensorflow:Assets written to: C:\Users\ricky\AppData\Local\Temp\tmpmm314l_b\assets


INFO:tensorflow:Assets written to: C:\Users\ricky\AppData\Local\Temp\tmpmm314l_b\assets


Saved artifact at 'C:\Users\ricky\AppData\Local\Temp\tmpmm314l_b'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1568), dtype=tf.float32, name='keras_tensor_102')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  2162547391760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2162547385232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2164653110096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2164653110672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2164653109904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2164653106832: TensorSpec(shape=(), dtype=tf.resource, name=None)


c:\Users\ricky\Desktop\KWS\venv\Lib\site-packages\tensorflow\lite\python\convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Modello salvato in: models1/best.tflite


In [15]:
# Controllo dimensione del modello quantizzato
import os
size_kb = os.path.getsize("models1/best.tflite") / 1024
print("Dimensione modello:", round(size_kb,2), "KB")

Dimensione modello: 9.41 KB


In [16]:
# Controllo dimensione del modello quantizzato
import os
size_kb = os.path.getsize("models1/best.tflite") / 1024
print("Dimensione modello:", round(size_kb,2), "KB")

Dimensione modello: 9.41 KB


In [17]:
# convert the tflite model to a C array (we use git bash on windows, so we can use xxd command)
# xxd -i best.tflite > model_data.cc

In [ ]:
# inside Arduino libraries folder on your system
# git clone https://github.com/tensorflow/tflite-micro-arduino-examples Arduino_TensorFlowLite